In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, losses
from tensorflow.keras.datasets import mnist

# 1. Prepare Normal Data (MNIST)
(x_train, _), (x_test, _) = mnist.load_data()
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

# 2. Build a Deep Autoencoder
class AnomalyDetector(models.Model):
    def __init__(self):
        super(AnomalyDetector, self).__init__()
        self.encoder = tf.keras.Sequential([
            layers.Flatten(),
            layers.Dense(64, activation="relu"),
            layers.Dense(32, activation="relu"),
            layers.Dense(16, activation="relu")]) # Tight bottleneck
        
        self.decoder = tf.keras.Sequential([
            layers.Dense(32, activation="relu"),
            layers.Dense(64, activation="relu"),
            layers.Dense(784, activation="sigmoid"),
            layers.Reshape((28, 28))])

    def call(self, x):
        encoded = self.encoder(x)
        return self.decoder(encoded)

autoencoder = AnomalyDetector()
autoencoder.compile(optimizer='adam', loss='mae') # Mean Absolute Error for loss

# Train ONLY on digits
autoencoder.fit(x_train, x_train, epochs=10, batch_size=256, validation_data=(x_test, x_test))

Epoch 1/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.2217 - val_loss: 0.1155
Epoch 2/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.1122 - val_loss: 0.1076
Epoch 3/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.1056 - val_loss: 0.1010
Epoch 4/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 0.0993 - val_loss: 0.0961
Epoch 5/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0951 - val_loss: 0.0946
Epoch 6/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.0942 - val_loss: 0.0937
Epoch 7/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0928 - val_loss: 0.0916
Epoch 8/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0910 - val_loss: 0.0905
Epoch 9/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 0.0900 - val_loss: 0.0888
Epoch 10/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0882 - val_loss: 0.0871


In [2]:
# Function to calculate reconstruction loss
def get_recon_loss(model, data):
    reconstructions = model.predict(data)
    return tf.keras.losses.mae(reconstructions.reshape(-1, 784), data.reshape(-1, 784)).numpy()

# Loss for a normal MNIST digit
normal_loss = get_recon_loss(autoencoder, x_test[:100])

# Simulating an anomaly (e.g., a Malayalam character or random noise)
# In a real scenario, you'd load an image of your language's letters here
anomaly_input = np.random.uniform(size=(1, 28, 28)) # Using noise as a placeholder anomaly
anomaly_loss = get_recon_loss(autoencoder, anomaly_input)

print(f"Average Normal Loss: {np.mean(normal_loss):.4f}")
print(f"Anomaly Loss: {np.mean(anomaly_loss):.4f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Average Normal Loss: 0.0809
Anomaly Loss: 0.4866


In [3]:
import cv2
import matplotlib.pyplot as plt

def load_malayalam_character(file_path):
    # 1. Load image in grayscale
    img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
    
    # 2. Resize to 28x28 (MNIST size)
    img = cv2.resize(img, (28, 28))
    
    # 3. Normalize to [0, 1]
    img = img.astype('float32') / 255.0
    
    # 4. CRITICAL: Invert colors. 
    # MNIST is white-on-black. If your image is black-on-white, you must flip it.
    img = 1.0 - img
    
    # 5. Add batch dimension for the model
    return np.expand_dims(img, axis=0)

# Replace with the filename in your VS Code sidebar
my_character = load_malayalam_character('mal3.png')

# 3. Calculate Loss
mal_loss = get_recon_loss(autoencoder, my_character)
print(f"Malayalam Character Reconstruction Loss: {np.mean(mal_loss):.4f}")

error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'resize'


In [4]:
reconstruction = autoencoder.predict(my_character)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title("Original (Malayalam)")
plt.imshow(my_character[0], cmap='gray')

plt.subplot(1, 2, 2)
plt.title(f"Reconstruction\nLoss: {np.mean(mal_loss):.4f}")
plt.imshow(reconstruction[0], cmap='gray')
plt.show()

NameError: name 'my_character' is not defined